# MPI COMMUNICATION:

## 1. INTRODUCTION:
MPI is an acronym for message passing protocol, is a standard for communication between multiple processes in parallel programming. 

In MPI, each process has its own private memory, therefore processes cannot access directly each other’s memory, though they exchange data through messages.

MPI is required in High performance computing, where complex program is distributed among multiple nodes, in order to reduce computational time.

There are several MPI versions available such MPI1, MPI2, MPI3, MPI4, in this study MPI4 will be discussed.

## 2. MPI4PY:
MPI4PY is a MPI binding for python programming language, it allows python program to exploit multiple processors.

In order to get start with mpi4py, first step is to install the library.

*python3 -m pip install mpi4py*

Then in a new python script, the library should be imported at the top of the script, before using mpi4.

from mpi4py import MPI

To start open parallel session following command is used;

*MPI.Init()*

At the end, after all the computations and exchange of data is finished, following command is used to close the parallel session.

*MPI.Finalize()*

How ever in mpi4py when library is imported both opening and closing of parallel session are done automatically.

**Communicator:** In parallel programming, communicator holds the group of processes, by default all the processes are in MPI.COMM. WORLD.

**Rank**, is the unique id associated to each process in a group, starting from zero. Whereas **size** refers to the total number of processes inside a group.

To get the rank of process id following command is used;  
*comm.Get_rank()*

To get the size of group, following command is used;  
*comm.Get_size()*

For sending and receiving messages between processes two types of communication used, blocking communication and non-blocking communication, either it’s a blocking communication or a non-blocking communication two types of communication functions are used;

**Lower Case:**  
**comm.send()** - used to send data in blocking communication.    
**comm.recv()** – used to receive data in blocking communication.  
**comm.isend()** – used to send data in non-blocking communication.  
**comm.irecv()** – used to receive data in non-blocking communication.  

In lower case communication function, data is sent as python object, mpi4py automatically convert object to bytes, this is known is pickling. On the receiving side the object is reconstructed, this is known as unpickling.

For every message this process is followed, this creates an overhead, and for large arrays, the process becomes slow.

**Upper Case:**    
**comm.Send()** - used to send data in blocking communication.  
**comm.Recv()** – used to receive data in blocking communication.  
**comm.Isend()** – used to send data in non-blocking communication.  
**comm.Irecv()** – used to receive data in non-blocking communication.  

The upper-case communication function works directly on memory buffer; therefore, it is much faster than the lower case.

## 3. File IO In MPI:  
MPI file IO is designed to let multiple process access same file simultaneously.
The mpi4py syntax is reported below;  
**File opening:**  
*fh = MPI.File.Open(comm, filename, amode)*    
*comm* – communicator.  
*filename* – name of file.  
*amode* – access mode.  
**access mode:**
*MPI.MODE_RDONLY* - Read only.  
*MPI.MODE_WRONLY* – Write only.  
*MPI.MODE_RDWR*    - Read & Write both.  
*MPI.MODE_CREATE* - Create file if it doesn't exist.  
*MPI.MODE_APPEND* -Append to file.  

**File closing:**  
*fh.Close()*

**File Writing:**  
*fh.Write(buffer)*  
*buffer* – NumPy array.  

**File Reading:**  
*fh.Read(buffer)*

To write read & Write at specific positions ***fh.Read_at(offset,buf)*** and  ***fh.Write_at(readat,buf)*** commands are used.

An MPI File IO example is reported below;

In [ ]:
# FILE IO MPI
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()

fh = MPI.File.Open(comm,"data.bin",MPI.MODE_CREATE | MPI.MODE_WRONLY)

data = np.array([my_id*10], dtype='i')

offset = my_id * data.itemsize

fh.Write_at(offset, data)

fh.Close()

## 4. Point to Point Communication:
It is the basic communication, that is performed between to processes that lie with in the same communicator. Process A sends the data and process B receives the data and vice versa. For point-to-point communication, both the source process and destination process must lie within same communicator. Within communicator source and destination processes are identified by their ranks.

There are two types of communication within the umbrella of point-to-point communication, that are;    
a.	Blocking communication.  
b.	Non-Blocking communication.  

**4.1. Blocking Communication:**  
In blocking communication, sender has to wait until the data is sent before reusing the buffer or performing other calculations. Similarly, receiver has also to wait until the data is received, this can cause allocated computational resources to go waste.

The communication can be further divided into two modes;  
a.	Synchronous    
b.	Standard or A synchronous  

In **synchronous communication**, the message is transferred directly from senders’ application buffer to receivers’ application buffer, the send operations are completed only when the matching receives has been initiated.

On the contrary, in **standard or Asynchronous communication mode**, message is copied into system internal buffer, send operation is completed when the message is safely copied into system internal buffer even if the receive operation is still not started. Then it’s the system buffer, whose responsibility is to transfer the data to receiver memory, when matching receive is posted.

For blocking, standard point to point communication, command syntax is reported below;

**Blocking send:**  
*comm.Send(buf, dest, tag)*  
*buf*- NumPy array or buffer containing data to send.  
*dest*- rank of destination process.  
*tag*- message identifier.  

**Blocking Receive:**  
*comm.recv(buf,source,tag,status)*    
*buf*- buffer to store incoming data.  
*source*- rank of source process.  
*tag*- message identifier.  
*status* – incoming message information.  
To read status information, following commands can be used;  
*status.Get_source()*      
*status.Get_tag()*    
*status.Get_error()*    
In mpi4py there is no error code argument, it means it used python exception for error handling.

An example of send and receive data between two processes using blocking communication is reported below;

In [ ]:
# P2P Blocking Communication
import numpy as np
from mpi4py import MPI


comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()

if my_id == 0:
    status = MPI.Status()
    A = np.array(2.0, dtype='d')
    B = np.empty(1, dtype='d')
    comm.Send(A,1,10)
    comm.Recv(B,1,11,status)
    print(f"My rank is {my_id} & I sent data  to process 1 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")
    print(f"Message source = {status.Get_source()}")
    print(f"Message tag    = {status.Get_tag()}")

elif my_id == 1:
    status = MPI.Status()
    B = np.empty(1, dtype='d')
    A = np.array(3.0, dtype='d')
    comm.Recv(B,0, 10,status=status)
    comm.Send(A,0,11)
    print(f"My rank is {my_id} & I sent data  to process 0 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")
    print(f"Message source = {status.Get_source()}")
    print(f"Message tag    = {status.Get_tag()}")


zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -n 2 python P2P.py  
My rank is 0 & I sent data  to process 1 sucessfully  
My rank is 1 & I sent data  to process 0 sucessfully  
my rank is 1: and i received Data = [2.]  
Message source = 0  
Message tag    = 10  
my rank is 0: and i received Data = [3.]  
Message source = 1  
Message tag    = 11  

There is also *comm.Sendrecv()*, which is a blocking communication, it combines send and receive in a single call. By blocking it means that until both send and receive operations are completed, program does not continue execution. Syntax is reported below;

*comm.Sendrecv(sendbuf, dest, sendtag, recvbuf, source, recvtag, status=status)*

**4.2. Non-Blocking Communication:**  
In non-blocking communication, sender and receive almost immediately, they do not wait communication events to complete.  
**Sender side:** 
a.	Send message using *Isend()*  
b.	Immediately continue executing other code.  
c.	Do computations  
d.	Wait for send to finish (using wait())  

**Receiver Side:**  
a.	Post for receiving (*IRecv()*)  
b.	Do computation  
c.	Wait for message to arrive  
d.	Use the data in message for other computations.  

In mpi4py *request.wait()* is used track the communication. It makes sure that non-blocking communication has finished.

*Wait()*, is used for single operation to be finished, for multiple communications waitll() is used.

**Non-Blocking Send:**  
*req = comm.Isend(buf, dest, tag)*  
*buf-* NumPy array or buffer containing data to send.  
*dest-* rank of destination process.  
*tag-* message identifier.  

Where *req* refers to request which is later used to trach and complete non-blocking communication.

*req.wait()*

**Non-Blocking Receive:**  
*req = comm.Irecv(buf, source, tag)*  
*buf*- buffer to store incoming data.  
*source*- rank of source process.  
*tag*- message identifier.  

Where req refers to request which is later used to trach and complete non-blocking communication.

*req.wait()*

An example of code of non blocking point to point communication is reported below;


In [ ]:
#P2P Non Blocking Communication.
import numpy as np
from mpi4py import MPI


comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()

if my_id == 0:
    status = MPI.Status()
    A = np.array(2.0, dtype='d')
    B = np.empty(1, dtype='d')
    req1 = comm.Isend(A,1,10)
    req2 = comm.Irecv(B,1,11)
    MPI.Request.Waitall([req1, req2])
    print(f"My rank is {my_id} & I sent data  to process 1 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")

elif my_id == 1:
    status = MPI.Status()
    B = np.empty(1, dtype='d')
    A = np.array(3.0, dtype='d')
    req1 = comm.Isend(A,0,11)
    req2 = comm.Irecv(B,0, 10)
    MPI.Request.Waitall([req1, req2])
    print(f"My rank is {my_id} & I sent data  to process 0 sucessfully")
    print(f"my rank is {my_id}: and i received Data = {B}")



zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -n 2 python P2PNonBlock.py  
My rank is 0 & I sent data  to process 1 sucessfully  
My rank is 1 & I sent data  to process 0 sucessfully  
my rank is 0: and i received Data = [3.]  
my rank is 1: and i received Data = [2.]  

## 5. Comparison between Fortran and python - P2P:
| Operation             | Fortran MPI        | Python mpi4py              |
|----------------------|--------------------|----------------------------|
| Blocking send        | `MPI_SEND`         | `comm.send()`              |
| Blocking receive     | `MPI_RECV`         | `comm.recv()`              |
| Non-blocking send    | `MPI_ISEND`        | `comm.Isend()`             |
| Non-blocking receive | `MPI_IRECV`        | `comm.Irecv()`             |
| Wait single          | `MPI_WAIT`         | `req.wait()`               |
| Wait multiple        | `MPI_WAITALL`      | `MPI.Request.Waitall()`    |
| Status               | Output argument    | `MPI.Status()` object      |
| Request handling     | Passed by reference| Returned object            |

## 6. Collective Communication:  
Collective communication takes place among all the process in a communicator, unlike point-to-point communication, which is held only between two processes at a time.  
The concepts used in collective communication are discussed below;

**6.1.	MPI Barrier:** it lets all the process to synchronize themselves with each other, it stops all the process until they are synchronized. It is useful in cases, where all the processes need to be at same point in a program. In mpi4py following command is used;  

*Comm.Barrier()*

**6.2.	MPI Broadcast:**  
When data is sent from one process to all the other processes in a communicator, this type of communication is known as broadcast, it means One to All. The sender process is known as root. There could be applications where a root process reads the data from input file, and broadcast this data to all the other processes in a communicator for further computations.

The syntax used in mpi4py is reported below;

*comm.Bcast(buff,root)*  
*buff -* buffer containing data to be send.  
*root –* Rank of the root process. 

An example is reported below;

In [ ]:
# MPI BCast
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()
root = 0

if my_id == 0:
   data = np.array([3,4],dtype='i')
else:
    data = np.empty(2, dtype='i')

print(f"Before BroadCast Rank:{my_id} & Data:{data}")
comm.Bcast(data, root=0)
print(f"After BroadCast Rank:{my_id} & Data:{data}")

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
Before BroadCast Rank:0 & Data:[3 4]  
Before BroadCast Rank:1 & Data:[1065353216 1065353216]  
After BroadCast Rank:0 & Data:[3 4]  
After BroadCast Rank:1 & Data:[3 4]  

**6.3.	MPI SCATTER**  
In this type of communication, the root process splits the data into ‘n’ equal segments, and distribute the data equally, it means each process in a communicator receives chunk of a data.

The syntax used in mpi4py is reported below;

*comm.Scatter(sendbuf, recvbuf, root)*  
*sendbuf* – buffer containing data to be send.  
*recvbuf* - receive buffer on each process.  
*root* – Rank of the root process.  

Another case could be that data to be distributed is of variable size, it means each process received variable size of data (chunk). For this purpose, scatterv is used. The syntax used in mpi4py is reported below;

*comm.Scatterv( [sendbuf, counts, displs, datatype], recvbuf, root)*  
*sendbuf* – buffer containing array of data to be sent.  
*counts* – size of data sent to each rank.  
*displs* – starting index of each chunk.  
*datatype* – specify the data type.  
*recvbuf* – receiving data buffer.    
*root* – rank of the root process. 

An example is reported below;

In [ ]:
# MPI Scatter
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
nproc = comm.Get_size()
root = 0

if my_id == 0:
    sendbuf = np.array([2,3], dtype='i')
else:
    sendbuf = None

recvbuf = np.empty(1, dtype='i')

comm.Scatter(sendbuf, recvbuf, root=0)

print(f"My id is {my_id} and i received {recvbuf}")


zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
My id is 1 and i received [3]  
My id is 0 and i received [2]  

In [ ]:
# MPI Scatterv
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
root = 0
if my_id == 0:
    sendbuf = np.array([2,3,5,6], dtype='i')
    counts = [1,3]
    displs = [0,1]
else:
    sendbuf = None
    counts = None
    displs = None

recvbuf = np.empty([1,3][my_id], dtype='i')

comm.Scatterv(
    [sendbuf, counts, displs, MPI.INT],
    recvbuf,
    root
)

print(f"I am rank {my_id}: and i received data {recvbuf}")

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
I am rank 0: and i received data [2]  
I am rank 1: and i received data [3 5 6]  

**6.4.	MPI GATHER:**  
In this type of communication, root process is used to collect data from all the other processes. The syntax used in mpi4py is reported below;

*comm.Gather(sendbuf, recvbuf, root)*  
*sendbuf* – local data chunk on each process.  
*recvbuf* – data buffer on root process.  
*root* - rank of the root process.  

The command syntax reported above is used to collect equal size of data chunks from each process, to received variable size of data chunks from each process **gatherv** is used. The syntax used in mpi4py is reported below;

*comm.Gatherv(sendbuf,[recvbuf, counts, displs, datatype], root)*  
*sendbuf* – data buffer on each local process.  
*recvbuf* – data buffer on root process.  
*counts* – size of data received from each chunk  
*datatype* – specify the datatype.    
*root* – rank of the root process. 

An example is reported below;

In [ ]:
# MPI Gather
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
root = 0
sendbuf = np.array([my_id], dtype='i')

if my_id == 0:
    recvbuf = np.empty(2, dtype='i')
else:
    recvbuf = None

comm.Gather(sendbuf, recvbuf, root)

if my_id == 0:
    print(f"i am rank {my_id} and i received the data {recvbuf}")


zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
i am rank 0 and i received the data [0 1]

In [ ]:
# MPI Gatherv
from mpi4py import MPI
import numpy as np
comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
root = 0
sendbuf = np.arange(my_id + 2, dtype='i')
counts = [2,3]
displs = [0,2]

if my_id == 0:
    recvbuf = np.empty(sum(counts), dtype='i')
else:
    recvbuf = None

comm.Gatherv(sendbuf,[recvbuf, counts, displs, MPI.INT],root)

if my_id == 0:
    print(f"i am rank {my_id} and i received the data {recvbuf}")

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
i am rank 0 and i received the data [0 1 0 1 2]

**6.5.	MPI ALL GATHER:**  
Unlike the gather operation, in which roots collect the data from each process, in **allgather** each process receives the final data.
The syntax used in mpi4py is reported below;

*comm.Allgather(sendbuf,recvbuf)*  
*sendbuf* – data buffer on each local process.  
*recvbuf* – gathered data on each process.  


In [ ]:
# MPI All Gather
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
sendbuf = np.array([my_id], dtype='i')
recvbuf = np.empty(comm.Get_size(), dtype='i')

comm.Allgather(sendbuf, recvbuf)

print(f"I am Rank {my_id}: & i received the data {recvbuf}")
   

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
I am Rank 1: & i received the data [0 1]  
I am Rank 0: & i received the data [0 1]  

**6.6.	MPI REDUCE:**  
The operation is used to perform reduction, data is collected from each process, a single process, root performs the reduction operation, i.e. sum or average, and stores the result.

The syntax used in mpi4py is reported below;

*comm.Reduce(sendbuf,recvbuf,op,root)*  
*sendbuf* - data send by each process.  
*recvbuf* - receive buffer on root process.  
*op* – reduction operation.  
*root* – rank of the root process.  

An example is reported below;

In [ ]:
# MPI Reduce
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
sendbuf = np.array(my_id + 2, dtype='i')

if my_id == 0:
    recvbuf = np.array(0, dtype='i')
else:
    recvbuf = None

comm.Reduce(sendbuf, recvbuf,op=MPI.SUM,root=0)

if my_id == 0:
    print(f"i am rank {my_id} i receive data {recvbuf}")
   

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
i am rank 0 i receive data 5

**6.7.	MPI ALLREDUCE:**  
Unlike the reduce, where final result is store on root process, in allreduce final result is distributed to all the processes in a communicator.  
The syntax used in mpi4py is reported below;

*comm.Allreduce(sendbuf,recvbuf,op)*  
*sendbuf* – local data buffer of each process.  
*recvbuf* - result data buffer on each process.  
*op* – reduction operation.  

An example is reported below;

In [ ]:
# MPI All Reduce
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
sendbuf = np.array(my_id + 2, dtype='i')
recvbuf = np.array(0, dtype='i')

comm.Allreduce(
    sendbuf,
    recvbuf,
    op=MPI.SUM
)

print(f"I am Rank {my_id}:& i received the data{recvbuf}")
   

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
I am Rank 1:& i received the data5  
I am Rank 0:& i received the data5

**6.8.	MPI ALL to ALL:**  
In this type of communication, data is exchanged among all the processes. Each process sends distinct data to every other process and receives the distinct data from every other process.  
The syntax used in mpi4py is reported below;

*comm.Alltoall(sendbuf,recvbuf)*  
*sendbuf* - data array buffer containing data for all the ranks.  
*recvbuf* – buffer array receiving data from all the ranks.  

An example is reported below;

In [ ]:
# MPI All to ALl
from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
my_id = comm.Get_rank()
size= comm.Get_size()

sendbuf = np.arange(my_id*size,(my_id+1)*size,dtype='i')

recvbuf = np.empty(size, dtype='i')

comm.Alltoall(sendbuf, recvbuf)

print(f"I am Rank {my_id} & i received data {recvbuf}")
   

zia__@DESKTOP-OVKLL81:~/jupyter$ mpirun -np 2 python collectivecomm.py  
I am Rank 0 & i received data [0 2]  
I am Rank 1 & i received data [1 3]  

# 7. Collective Communication Comparison:   Fortran MPI vs Python mpi4py

| Operation | Fortran MPI | Python mpi4py (Uppercase) |
|------------|------------|------------|
| Barrier | `MPI_BARRIER` | `comm.Barrier()` |
| Broadcast | `MPI_BCAST` | `comm.Bcast()` |
| Scatter | `MPI_SCATTER` | `comm.Scatter()` |
| Scatter Variable | `MPI_SCATTERV` | `comm.Scatterv()` |
| Gather | `MPI_GATHER` | `comm.Gather()` |
| Gather Variable | `MPI_GATHERV` | `comm.Gatherv()` |
| Reduce | `MPI_REDUCE` | `comm.Reduce()` |
| All Reduce | `MPI_ALLREDUCE` | `comm.Allreduce()` |
| All Gather | `MPI_ALLGATHER` | `comm.Allgather()` |
| All-to-All | `MPI_ALLTOALL` | `comm.Alltoall()` |

Uppercase mpi4 methods are close in performace to Fortran MPI subroutines

# 8. References:  
1. "https://mpi4py.readthedocs.io/en/stable/"
